# Credit / interest-rate link — basic checks

Small notebook to support the **assumptions** section of the HW5 presentation: we assume housing prices are related to credit conditions (proxied by the 30-year mortgage rate). Here we compute a few numbers we can report.

Uses the same data and date range as the main project (Boston, MA; 2010–2017 for consistency with the train period).

In [2]:
print("hello")

hello


In [3]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from scipy import stats

In [4]:
# Load data (same as main analysis)
ZDF = pd.read_csv('./Data/data_zillow_house_prices.csv')
IDF = pd.read_csv('./Data/data_interest_rates.csv')

# Boston, MA; 2010-01 to 2017-12 (train period)
BDF = ZDF[ZDF["RegionName"] == "Boston, MA"]
TS_BDF = BDF.drop(columns=["RegionID", "SizeRank", "RegionName", "RegionType", "StateName"])
values = TS_BDF.loc[11]
ts_df = pd.DataFrame(values)
ts_df.index = pd.to_datetime(ts_df.index)
ts_df = ts_df.loc["2010-01-01":"2017-12-31"]

IDF['DATE'] = pd.to_datetime(IDF['DATE'])
IDF = IDF.set_index('DATE')
rates_monthly = IDF['MORTGAGE30US'].resample('ME').mean().dropna()
date_start, date_end = "2010-01-01", "2017-12-31"
rates_aligned = rates_monthly.loc[date_start:date_end]
idx = pd.date_range(start=date_start, end=date_end, freq='ME')
rates_aligned = rates_aligned.reindex(idx).ffill().bfill()

price = np.asarray(ts_df.squeeze()).flatten()
rate = np.asarray(rates_aligned.values).flatten()
n = len(price)
assert len(rate) == n, "Length mismatch"
print(f"Sample: {n} months (2010-01 to 2017-12), Boston, MA")

Sample: 96 months (2010-01 to 2017-12), Boston, MA


## 1. Change in house prices and interest rate

**Δ price** = month-over-month change in house price. If credit matters, we might see negative correlation: higher rates → lower price growth (or vice versa).

In [5]:
delta_price = np.diff(price)  # length n-1
rate_level = rate[1:]          # align: rate in month t with Δprice from t-1 to t
delta_rate = np.diff(rate)    # length n-1

r_level, p_level = stats.pearsonr(delta_price, rate_level)
r_diff, p_diff = stats.pearsonr(delta_price, delta_rate)

print("Correlation (report these):")
print(f"  corr(Δ price, interest rate level) = {r_level:.4f}  (p = {p_level:.4f})")
print(f"  corr(Δ price, Δ interest rate)     = {r_diff:.4f}  (p = {p_diff:.4f})")
print()
print("Interpretation: Negative correlation with rate level means higher mortgage rate tends to go with smaller (or negative) monthly price change.")

Correlation (report these):
  corr(Δ price, interest rate level) = -0.4044  (p = 0.0000)
  corr(Δ price, Δ interest rate)     = 0.1667  (p = 0.1065)

Interpretation: Negative correlation with rate level means higher mortgage rate tends to go with smaller (or negative) monthly price change.


## 2. Simple regression: Δ price on interest rate

Regression **Δ price = α + β × (interest rate)**. Report β (and its p-value) and R².

In [6]:
X = sm.add_constant(rate_level)
model = sm.OLS(delta_price, X).fit()
print(model.summary())
print()
print("Numbers to report:")
print(f"  Slope (β): {model.params[1]:.2f}  (change in $/month per 1 p.p. rate)")
print(f"  p-value for rate: {model.pvalues[1]:.4f}")
print(f"  R²: {model.rsquared:.4f}")

                            OLS Regression Results                            
Dep. Variable:                      y   R-squared:                       0.164
Model:                            OLS   Adj. R-squared:                  0.155
Method:                 Least Squares   F-statistic:                     18.18
Date:                Wed, 18 Mar 2026   Prob (F-statistic):           4.82e-05
Time:                        20:49:48   Log-Likelihood:                -817.42
No. Observations:                  95   AIC:                             1639.
Df Residuals:                      93   BIC:                             1644.
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const       6740.9755   1317.943      5.115      0.0

## 3. Detrended price vs interest rate (tie to ARX)

Same scale as in the main model: detrended price (residual after linear trend). Correlation and simple regression.

In [7]:
T = len(price)
X_trend = sm.add_constant(np.arange(T))
ols_trend = sm.OLS(price, X_trend).fit()
detrended = ols_trend.resid

r_det, p_det = stats.pearsonr(detrended, rate)
print("Correlation (detrended price, interest rate):")
print(f"  r = {r_det:.4f},  p = {p_det:.4f}")
print()

X_reg = sm.add_constant(rate)
model_det = sm.OLS(detrended, X_reg).fit()
print("Regression: detrended price ~ interest rate")
print(f"  Coef (rate): {model_det.params[1]:.2f},  p = {model_det.pvalues[1]:.4f},  R² = {model_det.rsquared:.4f}")

Correlation (detrended price, interest rate):
  r = 0.5803,  p = 0.0000

Regression: detrended price ~ interest rate
  Coef (rate): 19289.69,  p = 0.0000,  R² = 0.3368


---
**For the presentation:** Pick one or two numbers (e.g. correlation and its p-value, or slope and R²) and say: "We tested the link between credit conditions and house prices. For example, the correlation between month-over-month change in Boston house prices and the 30-year mortgage rate was r = … (p = …), consistent with the idea that interest rates help explain variation in prices, which motivated including the rate as an exogenous variable in our ARX/ARMAX models."